# Python Engineering for AI Workflows

This notebook covers the full set of Python engineering practices behind
real AI systems: core language fundamentals, functions and classes, file
and data handling, error handling, logging, configuration, async basics,
type hints and clean-code tooling, and reproducibility — finishing with
the standard project structure that ties everything together.

Every topic follows the same pattern: a plain explanation, a short
example, and why it matters for AI/ML work. Run the cells in order from
top to bottom.


## Setup

Run this cell first. It imports everything used later in the notebook.


In [1]:
import os
import json
import time
import random
import asyncio
import logging
import subprocess
from pathlib import Path
from dataclasses import dataclass
from typing import Optional, List, Dict, Union

import yaml
import pandas as pd
from dotenv import load_dotenv
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier

print("Setup complete.")


Setup complete.


## 1. Python Programming Fundamentals

Python is the most common language for AI/ML work because it is readable,
has a huge ecosystem (pandas, scikit-learn, PyTorch, LangChain), and is
flexible enough to go from a quick experiment to a production system.

This section is a fast refresher on the basics that every later topic
builds on: variables and types, control flow, core data structures, and
string formatting.


### Variables and data types

Every value in Python has a type. AI/ML code constantly moves data between
these types, so it helps to be precise about them.

- `int` / `float` — numbers, such as an epoch count or a learning rate
- `str` — text, such as a file path or a label
- `bool` — `True` / `False`, such as a flag like `use_gpu`
- `None` — means "no value yet"


In [2]:
# Example: a small set of model hyperparameters
learning_rate = 0.001      # float
epochs = 50                # int
model_name = "random_forest"  # str
use_gpu = False             # bool
checkpoint_path = None     # not set yet

print(type(learning_rate), type(epochs), type(model_name), type(use_gpu))


<class 'float'> <class 'int'> <class 'str'> <class 'bool'>


### Control flow: if/else and loops

Control flow lets code make decisions (`if` / `elif` / `else`) and repeat
work (`for`, `while`). This is how you filter, transform, and process data.


In [3]:
# Example: keep only the "passing" scores from a list
scores = [91, 42, 77, 38, 95]
passing = []

for score in scores:
    if score >= 60:
        passing.append(score)
    else:
        continue

print("Passing scores:", passing)


Passing scores: [91, 77, 95]


### Core data structures

Four containers cover almost everything in day-to-day Python:

- `list` — ordered, changeable: `[91, 84, 77]`
- `tuple` — ordered, unchangeable: `("train", "test")`
- `dict` — key-value pairs: `{"lr": 0.001, "epochs": 50}`
- `set` — unique, unordered values: `{"cat", "dog"}`


In [4]:
# Example: a small "batch" of labeled records
batch = [
    {"id": 1, "label": "cat"},
    {"id": 2, "label": "dog"},
    {"id": 3, "label": "cat"},
]

unique_labels = {record["label"] for record in batch}
print("Unique labels:", unique_labels)


Unique labels: {'dog', 'cat'}


### Strings and f-strings

f-strings (`f"..."`) let you embed variables directly inside a string.
They are the cleanest way to build log messages, file names, and prompts.


In [5]:
# Example: building a log-style message
epoch = 12
loss = 0.034215

message = f"Epoch {epoch}: loss={loss:.4f}"
print(message)


Epoch 12: loss=0.0342


**Quick drill:** using only what you've seen so far, write a small script
that keeps rows with `score >= 60` from a list of dicts, collects the
unique labels among the kept rows, and prints a summary using an f-string.


In [6]:
records = [
    {"score": 91, "label": "cat"},
    {"score": 42, "label": "dog"},
    {"score": 77, "label": "cat"},
    {"score": 38, "label": "bird"},
    {"score": 95, "label": "dog"},
]

kept = [r for r in records if r["score"] >= 60]
labels_seen = {r["label"] for r in kept}

print(f"Kept {len(kept)} of {len(records)} records. Labels seen: {labels_seen}")


Kept 3 of 5 records. Labels seen: {'dog', 'cat'}


## 2. Functions and Modularity

Repeating the same logic in multiple places is fragile: fix a bug in one
copy and forget the other three. Functions solve this by giving a piece of
logic one name and one place to live.


### Before and after: why functions matter


In [7]:
# Before: the same calculation repeated three times
correct1, total1 = 91, 100
correct2, total2 = 77, 100
correct3, total3 = 95, 100

acc1 = correct1 / total1
print(f"Model A: {acc1:.2%}")

acc2 = correct2 / total2
print(f"Model B: {acc2:.2%}")

acc3 = correct3 / total3
print(f"Model C: {acc3:.2%}")


Model A: 91.00%
Model B: 77.00%
Model C: 95.00%


In [8]:
# After: one function, reused three times
def report_accuracy(name, correct, total):
    accuracy = correct / total
    print(f"{name}: {accuracy:.2%}")

report_accuracy("Model A", correct1, total1)
report_accuracy("Model B", correct2, total2)
report_accuracy("Model C", correct3, total3)


Model A: 91.00%
Model B: 77.00%
Model C: 95.00%


### Anatomy of a function

A function has a name, parameters (some with default values), a body, and
a `return` value. A docstring (the text in triple quotes right after the
`def` line) documents what the function does.


In [9]:
def train_model(data, epochs=10, lr=0.001):
    """Pretend to train a model and return a fake final loss."""
    final_loss = 1.0 / (epochs * (1 + lr))
    return final_loss

final_loss = train_model(data=None, epochs=20)
print(f"Final loss: {final_loss:.4f}")


Final loss: 0.0500


### `*args` and `**kwargs`

Sometimes a function needs to accept a flexible number of arguments.
`*args` collects extra positional arguments into a tuple, and `**kwargs`
collects extra keyword arguments into a dict. This is common in logging
wrappers and other general-purpose helper functions.


In [10]:
def log_event(event_name, *args, **kwargs):
    print(f"[{event_name}] args={args} kwargs={kwargs}")

log_event("training_start", 50, lr=0.001, use_gpu=False)


[training_start] args=(50,) kwargs={'lr': 0.001, 'use_gpu': False}


### Organizing code into modules

Once you have a few related functions, the next step is moving them out of
a notebook and into a `.py` file (a *module*), so they can be imported and
reused anywhere. We will do exactly this later in the **Project Structure**
section, moving functions defined here into `project/src/`.


## 3. Classes and Object-Oriented Programming

Many AI/ML libraries (scikit-learn, PyTorch) are built around classes.
Understanding classes means you can read their source code, and design
your own reusable components the same way.

A **class** is a blueprint. An **object** is a specific instance created
from that blueprint, holding its own data.


In [11]:
class Dataset:
    def __init__(self, name, rows):
        self.name = name
        self.rows = rows

    def describe(self):
        return f"{self.name}: {self.rows} rows"

train = Dataset("train", 1000)
test = Dataset("test", 200)

print(train.describe())
print(test.describe())


train: 1000 rows
test: 200 rows


### Inheritance and polymorphism

Inheritance lets one class reuse and extend another, instead of copying
code. A common pattern is a `BaseModel` that defines a shared interface,
with each specific model type implementing its own version of a method
(this is called polymorphism).


In [12]:
class BaseModel:
    def train(self, X, y):
        raise NotImplementedError("Subclasses must implement train()")


class RandomForestModel(BaseModel):
    def __init__(self, n_estimators=100, random_state=42):
        self.n_estimators = n_estimators
        self.random_state = random_state
        self.model = None

    def train(self, X, y):
        self.model = RandomForestClassifier(
            n_estimators=self.n_estimators,
            random_state=self.random_state,
        )
        self.model.fit(X, y)
        return self.model


X_demo, y_demo = make_classification(n_samples=100, n_features=4, random_state=42)
rf_model = RandomForestModel(n_estimators=50)
rf_model.train(X_demo, y_demo)
print("Trained model:", rf_model.model)


Trained model: RandomForestClassifier(n_estimators=50, random_state=42)


### `@dataclass` for clean configuration objects

Writing `__init__` by hand for simple data-holding classes gets repetitive.
The `@dataclass` decorator generates it for you automatically, and works
well together with type hints (covered later in this notebook).


In [13]:
@dataclass
class TrainingConfig:
    learning_rate: float = 0.001
    epochs: int = 50
    model_name: str = "random_forest"

config = TrainingConfig(epochs=100)
print(config)
print("Epochs:", config.epochs)


TrainingConfig(learning_rate=0.001, epochs=100, model_name='random_forest')
Epochs: 100


## 4. File Handling

Almost every AI pipeline starts by reading a file and ends by writing one.
This section covers reading and writing files safely, and using `pathlib`
to build file paths that work on any operating system.


In [14]:
# Create a small working folder for this notebook's examples
Path("notebook_demo").mkdir(exist_ok=True)

# Writing a plain text file
with open("notebook_demo/sample_log.txt", "w") as f:
    f.write("INFO: training started\n")
    f.write("ERROR: missing feature column\n")
    f.write("INFO: training finished\n")

# Reading it back, line by line, filtering for errors
with open("notebook_demo/sample_log.txt") as f:
    for line in f:
        if "ERROR" in line:
            print("Found:", line.strip())


Found: ERROR: missing feature column


### `pathlib` for cross-platform paths

A string path like `"data\\file.csv"` only works on Windows, and
`"data/file.csv"` written by hand is easy to get wrong. `pathlib.Path`
builds paths correctly regardless of operating system.


In [15]:
data_dir = Path("notebook_demo") / "data"
data_dir.mkdir(parents=True, exist_ok=True)

file_path = data_dir / "train.csv"
print("Path:", file_path)
print("Exists yet?", file_path.exists())


Path: notebook_demo\data\train.csv
Exists yet? False


### Working with CSV data

`pandas` is the standard tool for reading and writing tabular data such as
CSV files — this is how you will usually load a dataset or save
predictions.


In [16]:
# Generate a small synthetic dataset and save it as CSV
X, y = make_classification(n_samples=200, n_features=5, random_state=42)
df = pd.DataFrame(X, columns=[f"feature_{i}" for i in range(5)])
df["label"] = y

df.to_csv(data_dir / "train.csv", index=False)
print(f"Saved {len(df)} rows to {data_dir / 'train.csv'}")
df.head()


Saved 200 rows to notebook_demo\data\train.csv


,feature_0,feature_1,feature_2,feature_3,feature_4,label
0,0.407719,-0.095667,1.502357,-1.209659,-1.277852,0
1,1.171938,-0.734462,0.311250,-0.052759,-2.182368,1
2,0.208795,0.045808,0.332314,-1.325960,-0.961946,0
3,-0.766396,0.297780,-0.351513,1.394775,2.019333,1
4,-0.474432,0.295337,1.676437,0.036211,0.889946,1


## 5. JSON and YAML Handling

JSON and YAML are the two most common formats for structured, non-tabular
data in AI/ML work: JSON for things like metrics and API payloads, and YAML
for configuration files, because it is easier for humans to read and edit.


In [17]:
# JSON: good for saving results / metrics
metrics = {"accuracy": 0.94, "f1_score": 0.91}

with open("notebook_demo/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

with open("notebook_demo/metrics.json") as f:
    loaded_metrics = json.load(f)

print(loaded_metrics)


{'accuracy': 0.94, 'f1_score': 0.91}


In [18]:
# YAML: good for configuration
config_dict = {
    "model": {"type": "random_forest", "n_estimators": 200},
    "training": {"test_size": 0.2, "random_state": 42},
}

with open("notebook_demo/config.yaml", "w") as f:
    yaml.dump(config_dict, f, sort_keys=False)

print(open("notebook_demo/config.yaml").read())


model:
  type: random_forest
  n_estimators: 200
training:
  test_size: 0.2
  random_state: 42



Notice the difference: JSON uses braces and quotes, while YAML uses plain
indentation. Both map directly to Python dictionaries once loaded —
use `json.load` / `json.dump` for JSON, and `yaml.safe_load` / `yaml.dump`
for YAML. Always use `safe_load`, not `load`, since `safe_load` cannot
execute arbitrary code hidden inside a YAML file.


## 6. Exception Handling

Real data is messy: files go missing, rows are malformed, networks time
out. Without handling errors, one bad row can crash an entire pipeline.
Exception handling lets you respond to a specific problem deliberately,
instead of the whole program stopping unexpectedly.


### `try` / `except` / `else` / `finally`

- `try`: the code that might fail
- `except SomeError`: runs only if that specific error happens
- `else`: runs only if no exception occurred
- `finally`: always runs, useful for cleanup


In [19]:
def safe_load_csv(path):
    try:
        df = pd.read_csv(path)
    except FileNotFoundError:
        print(f"File not found: {path}")
        df = None
    else:
        print(f"Loaded {len(df)} rows from {path}")
    finally:
        print("Load attempt finished.")
    return df

# This one exists
df_ok = safe_load_csv(data_dir / "train.csv")
print()
# This one does not
df_missing = safe_load_csv("notebook_demo/does_not_exist.csv")


Loaded 200 rows from notebook_demo\data\train.csv
Load attempt finished.

File not found: notebook_demo/does_not_exist.csv
Load attempt finished.



Load attempt finished.

File not found: notebook_demo/does_not_exist.csv
Load attempt finished.


### Custom exceptions

Built-in exceptions do not always describe your specific problem. A custom
exception, created by subclassing `Exception`, makes errors in your own
code self-explanatory.


In [20]:
class DataValidationError(Exception):
    """Raised when input data fails a quality check."""
    pass


def validate(df):
    if df.isnull().values.any():
        raise DataValidationError("Dataset contains null values")
    return True


try:
    validate(df_ok)
    print("Data passed validation.")
except DataValidationError as e:
    print("Validation failed:", e)


Data passed validation.


## 7. Logging

`print()` disappears the moment your terminal or notebook session closes.
The `logging` module keeps a permanent, timestamped, filterable record of
what a program did — essential once a script runs unattended (for
example, an overnight training job).


In [21]:
Path("notebook_demo/logs").mkdir(exist_ok=True)

logger = logging.getLogger("demo_logger")
logger.setLevel(logging.INFO)
# Avoid duplicate handlers if this cell is re-run
logger.handlers.clear()

handler = logging.FileHandler("notebook_demo/logs/train.log")
handler.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s"))
logger.addHandler(handler)

logger.info("Training started")
logger.warning("Learning rate seems high")
logger.info("Training finished")

with open("notebook_demo/logs/train.log") as f:
    print(f.read())


2026-07-24 21:46:46,177 INFO Training started
2026-07-24 21:46:46,177 WARNING Learning rate seems high
2026-07-24 21:46:46,177 INFO Training finished



Log levels, in increasing order of severity, are `DEBUG`, `INFO`,
`WARNING`, `ERROR`, and `CRITICAL`. Setting `level=logging.INFO` means
`DEBUG` messages are ignored, but `INFO` and everything more severe is
recorded.


## 8. Configuration Management

Hardcoded values — file paths, hyperparameters, batch sizes —
mean that every change requires editing code. Config-driven workflows
separate **settings** from **logic**, so the same code can run in
different environments just by swapping a config file.

There are two kinds of settings, and they belong in two different places:

| | `config.yaml` | `.env` |
|---|---|---|
| Holds | Non-secret parameters (model type, paths, hyperparameters) | Secrets (API keys, tokens, passwords) |
| Committed to git? | Yes | Never — only `.env.example` is committed |
| Loaded with | `yaml.safe_load()` | `python-dotenv` → `os.getenv()` |


In [22]:
# We already wrote notebook_demo/config.yaml earlier — read it back
with open("notebook_demo/config.yaml") as f:
    loaded_config = yaml.safe_load(f)

print("Config:", loaded_config)


Config: {'model': {'type': 'random_forest', 'n_estimators': 200}, 'training': {'test_size': 0.2, 'random_state': 42}}


In [23]:
# .env holds secrets and is never committed to git.
# .env.example holds the same variable names with placeholder values,
# and is safe to commit so teammates know what they need to set.
with open("notebook_demo/.env", "w") as f:
    f.write("API_KEY=sk-demo-12345\n")

with open("notebook_demo/.env.example", "w") as f:
    f.write("API_KEY=your-api-key-here\n")

load_dotenv("notebook_demo/.env")
api_key = os.getenv("API_KEY", "not-set")

# Never print a secret in full — mask it, even in a demo
masked = api_key[:3] + "*" * max(len(api_key) - 3, 0)
print("API_KEY loaded:", masked)


API_KEY loaded: sk-**********


 sk-**********


`config.yaml` and `.env` work together: `config.yaml` describes **what**
the pipeline should do, and `.env` describes **who you are** (credentials).
Both are loaded the same way in real code — once at the start of a
script.


## 9. Async Programming Basics

Many AI workflows spend most of their time *waiting* — for an API
response, a file download, a database query. Async programming lets a
program overlap that waiting instead of doing it one task at a time. It
does not speed up CPU-heavy work like model training; it only helps when
a task is waiting on something external (this is called I/O-bound work).


### Synchronous version

Here, three "API calls" happen one after another. Each one waits a second,
so the total time is roughly the sum of all three waits.


In [24]:
def fetch_prediction_sync(prompt):
    time.sleep(1)  # simulates waiting on a network call
    return f"prediction for: {prompt}"

start = time.time()
results = [fetch_prediction_sync(p) for p in ["a", "b", "c"]]
elapsed = time.time() - start

print(results)
print(f"Sync elapsed: {elapsed:.1f} seconds")


['prediction for: a', 'prediction for: b', 'prediction for: c']
Sync elapsed: 3.0 seconds


### Asynchronous version

`async def` defines a coroutine, and `await` pauses it until something
finishes. `asyncio.gather()` runs several coroutines concurrently, so the
total time is roughly the length of the *longest* single wait, not the sum
of all of them.

Note: in a plain Python script you would normally write
`asyncio.run(main())` to start this. Inside a Jupyter notebook, an event
loop is already running, so we can simply `await` a coroutine directly in
a cell.


In [25]:
async def fetch_prediction_async(prompt):
    await asyncio.sleep(1)  # simulates waiting on a network call
    return f"prediction for: {prompt}"

async def fetch_all(prompts):
    tasks = [fetch_prediction_async(p) for p in prompts]
    return await asyncio.gather(*tasks)

start = time.time()
results = await fetch_all(["a", "b", "c"])
elapsed = time.time() - start

print(results)
print(f"Async elapsed: {elapsed:.1f} seconds")


['prediction for: a', 'prediction for: b', 'prediction for: c']
Async elapsed: 1.0 seconds


Both versions do the same three "calls", but the async version finishes in
roughly a third of the time, because the waiting overlaps instead of
stacking up.


## 10. Type Hints and Clean-Code Practices

Type hints document what a function expects and returns. They are not
enforced while the code runs, but they help catch mistakes early through
tools and editors, and make code easier for others to read.


In [26]:
# Without type hints, it's unclear what "data" or "epochs" should be
def train_untyped(data, epochs, lr):
    return {"epochs": epochs, "lr": lr}

# With type hints, the expectations are explicit
def train_typed(data: pd.DataFrame, epochs: int, lr: float) -> dict:
    return {"epochs": epochs, "lr": lr}

print(train_typed(df_ok, epochs=20, lr=0.001))


{'epochs': 20, 'lr': 0.001}


Common typing patterns from the `typing` module:

- `Optional[str]` — a value that might be `None`
- `List[float]`, `Dict[str, int]` — typed collections
- `Union[int, float]` — accepts either type


In [27]:
def load_data(path: str, columns: Optional[List[str]] = None) -> Dict[str, list]:
    """Load a dataset and optionally restrict it to specific columns."""
    df = pd.read_csv(path)
    if columns:
        df = df[columns]
    return {"rows": df.to_dict("records"), "columns": list(df.columns)}

result = load_data(str(data_dir / "train.csv"), columns=["feature_0", "label"])
print("Columns loaded:", result["columns"])


Columns loaded: ['feature_0', 'label']


### Naming and docstrings (PEP 8)

PEP 8 is Python's official style guide. Two of its most important rules:
use `snake_case` for variables and functions, and `PascalCase` for class
names. A short docstring explaining *what* a function does (not *how*) is
good practice for anything reused elsewhere.


### Linting and formatting with Black and Ruff

**Black** is an autoformatter: it rewrites your code into one consistent
style automatically, so nobody has to argue about spacing or quote style.
**Ruff** is a linter: it scans your code for real problems — unused
imports, undefined names, unreachable code — and can auto-fix many of
them. Together, they are the standard formatting and linting combination
for modern Python projects.

Let's see them work on a deliberately messy file.


In [28]:
messy_code = '''import os
import pandas as pd

def train(data,epochs=10,lr =0.001):
  unused_variable = 123
  model=data
  return model
'''

messy_path = Path("notebook_demo/messy_example.py")
messy_path.write_text(messy_code)

print("Before formatting:")
print(messy_path.read_text())


Before formatting:
import os
import pandas as pd

def train(data,epochs=10,lr =0.001):
  unused_variable = 123
  model=data
  return model



In [29]:
# Run Black to auto-format the file
result = subprocess.run(
    ["black", str(messy_path)],
    capture_output=True, text=True,
)
print("black stdout:", result.stdout.strip())
print("black stderr:", result.stderr.strip())

print()
print("After formatting:")
print(messy_path.read_text())


black stdout: 
black stderr: reformatted notebook_demo\messy_example.py

All done! \u2728 \U0001f370 \u2728
1 file reformatted.

After formatting:
import os
import pandas as pd


def train(data, epochs=10, lr=0.001):
    unused_variable = 123
    model = data
    return model



In [30]:
# Run Ruff to lint the (now formatted) file
result = subprocess.run(
    ["ruff", "check", str(messy_path)],
    capture_output=True, text=True,
)
print(result.stdout)


]8;;https://docs.astral.sh/ruff/rules/unsorted-imports\I001]8;;\ [*] Import block is un-sorted or un-formatted
 --> notebook_demo\messy_example.py:1:1
  |
1 | / import os
2 | | import pandas as pd
  | |___________________^
  |
help: Organize imports
  |
1 | import os
2 +
3 | import pandas as pd
  |

]8;;https://docs.astral.sh/ruff/rules/unused-import\F401]8;;\ [*] `os` imported but unused
 --> notebook_demo\messy_example.py:1:8
  |
1 | import os
  |        ^^
2 | import pandas as pd
  |
help: Remove unused import: `os`
  |
  - import os
1 | import pandas as pd
  |

]8;;https://docs.astral.sh/ruff/rules/unused-import\F401]8;;\ [*] `pandas` imported but unused
 --> notebook_demo\messy_example.py:2:18
  |
1 | import os
2 | import pandas as pd
  |                  ^^
  |
help: Remove unused import: `pandas`
  |
1 | import os
  - import pandas as pd
2 |
  |

]8;;https://docs.astral.sh/ruff/rules/unused-variable\F841]8;;\ Local variable `unused_variable` is assigned to but ne

Ruff correctly flags the unused `import os` and the unused
`unused_variable` — exactly the kind of small mistake that is easy to
miss by eye but easy for a linter to catch. In a real project, both
commands are usually run as `black .` and `ruff check .` from the project
root, checking every file at once.


## 11. Notebook vs Production Code

Notebooks and scripts are good at different things:

- **Notebooks** are best for exploration: you see output immediately,
  cell by cell, which is great for visualizing data. They are harder to
  test, version, or reuse.
- **Scripts** (`.py` files) are best for production: they run top to
  bottom with no hidden state, and can be tested, scheduled, and imported
  elsewhere.

The common pattern is to explore in a notebook first, then "graduate" the
working logic into a script once it is ready to run repeatedly.


### Refactor example

This messy notebook cell hardcodes a path and mixes loading, cleaning, and
training together in one block.


In [31]:
# Messy notebook-style cell
df = pd.read_csv(str(data_dir / "train.csv"))   # hardcoded path
df = df.dropna()
X = df.drop("label", axis=1)
y = df["label"]

model = RandomForestClassifier()
model.fit(X, y)
print("Training accuracy:", model.score(X, y))


Training accuracy: 1.0


The same logic, refactored into small, named, typed functions — ready
to be moved into a `.py` file:


In [32]:
def load_and_clean(path: str) -> pd.DataFrame:
    return pd.read_csv(path).dropna()

def split_features_target(df: pd.DataFrame, target_col: str = "label"):
    X = df.drop(target_col, axis=1)
    y = df[target_col]
    return X, y

def train(X, y, random_state: int = 42) -> RandomForestClassifier:
    model = RandomForestClassifier(random_state=random_state)
    model.fit(X, y)
    return model

df_clean = load_and_clean(str(data_dir / "train.csv"))
X, y = split_features_target(df_clean)
model = train(X, y)
print("Training accuracy:", model.score(X, y))


Training accuracy: 1.0


## 12. Reproducibility Practices

"It worked on my machine" is one of the most common problems in AI/ML
work. Reproducibility means the same code and the same data reliably
produce the same result — which is what makes results trustworthy and
comparable across runs.


In [33]:
# Without a fixed random_state, results can differ between runs
scores_without_seed = []
for _ in range(3):
    m = RandomForestClassifier()
    m.fit(X, y)
    scores_without_seed.append(round(m.score(X, y), 4))

print("Scores without a fixed seed:", scores_without_seed)


Scores without a fixed seed: [1.0, 1.0, 1.0]


In [34]:
# With a fixed random_state, the result is identical every time
scores_with_seed = []
for _ in range(3):
    m = RandomForestClassifier(random_state=42)
    m.fit(X, y)
    scores_with_seed.append(round(m.score(X, y), 4))

print("Scores with a fixed seed:", scores_with_seed)


Scores with a fixed seed: [1.0, 1.0, 1.0]


Three practices get you most of the way to reproducible results:

- Fix `random_state` (or an equivalent seed) everywhere randomness is used
- Pin exact library versions in `requirements.txt`
- Keep the config and logs from each run, so you know which settings
  produced which result

We'll put all of this together in the next section.


## 13. AI Project Structure

You've now seen functions, classes, file handling, JSON/YAML, exceptions,
logging, configuration, async, type hints, and reproducibility. Professional
AI teams don't scatter these across a folder randomly — they follow a
standard layout, so any teammate can open a project and immediately know
where things live.

```
project/
├── data/         raw & processed datasets
├── models/       saved model artifacts
├── notebooks/    exploration & prototyping
├── src/          production-ready code (functions, classes)
├── configs/      config.yaml — non-secret settings
├── logs/         logging output, run metadata
├── .env          secrets — never committed
└── .env.example  template — safe to commit
```

Each folder maps back to a topic from this notebook:

| Folder | Connects to |
|---|---|
| `data/` | Section 4 — File Handling |
| `models/` | Section 3 — Classes & OOP |
| `notebooks/` | Section 11 — Notebook vs Production |
| `src/` | Sections 2, 3, 10 — Functions, Classes, Type Hints/Clean Code |
| `configs/` | Section 8 — Configuration Management |
| `logs/` | Sections 7, 12 — Logging, Reproducibility |
| `.env` | Section 8 — Configuration Management |

Let's scaffold it for real.


In [35]:
# Create the standard folder structure
folders = ["data", "models", "notebooks", "src", "configs", "logs"]
for folder in folders:
    os.makedirs(os.path.join("project", folder), exist_ok=True)

print("Created project structure:")
for folder in folders:
    print(f"  project/{folder}/")


Created project structure:
  project/data/
  project/models/
  project/notebooks/
  project/src/
  project/configs/
  project/logs/


In [36]:
# Reuse the dataset from earlier and save it into the project's data/ folder
df.to_csv("project/data/train.csv", index=False)

# Config: non-secret settings
project_config = {
    "data_path": "data/train.csv",
    "model": {"type": "random_forest", "n_estimators": 200},
    "training": {"test_size": 0.2, "random_state": 42},
}
with open("project/configs/config.yaml", "w") as f:
    yaml.dump(project_config, f, sort_keys=False)

# Secrets: .env (never committed) and .env.example (safe to commit)
with open("project/.env", "w") as f:
    f.write("API_KEY=sk-demo-12345\n")
with open("project/.env.example", "w") as f:
    f.write("API_KEY=your-api-key-here\n")

# .gitignore: exclude data, models, logs, and secrets from version control
with open("project/.gitignore", "w") as f:
    f.write("data/\nmodels/\nlogs/\n__pycache__/\n*.pyc\n.env\n")

print("Wrote config.yaml, .env, .env.example, and .gitignore")


Wrote config.yaml, .env, .env.example, and .gitignore


Now let's write the production version of our training logic into
`project/src/train.py`. It brings together almost everything from this
notebook: type hints, a class, config loading, `.env` loading, exception
handling, logging, and a fixed random seed for reproducibility.


In [37]:
train_script = '''"""
train.py — production training script.
Run from the project/ root:  python src/train.py
"""
import logging
import os
import random

import pandas as pd
import yaml
from dotenv import load_dotenv
from sklearn.ensemble import RandomForestClassifier


class DataValidationError(Exception):
    """Raised when input data fails a quality check."""


class RandomForestModel:
    def __init__(self, n_estimators: int = 100, random_state: int = 42):
        self.n_estimators = n_estimators
        self.random_state = random_state
        self.model = None

    def train(self, X, y):
        self.model = RandomForestClassifier(
            n_estimators=self.n_estimators,
            random_state=self.random_state,
        )
        self.model.fit(X, y)
        return self.model


def load_config(path: str = "configs/config.yaml") -> dict:
    with open(path) as f:
        return yaml.safe_load(f)


def load_data(path: str) -> pd.DataFrame:
    try:
        df = pd.read_csv(path)
    except FileNotFoundError as e:
        raise FileNotFoundError(f"Could not find data file: {path}") from e
    df = df.dropna()
    if df.empty:
        raise DataValidationError("Dataset is empty after dropping missing values")
    return df


def main():
    logging.basicConfig(
        filename="logs/train.log",
        level=logging.INFO,
        format="%(asctime)s %(levelname)s %(message)s",
    )
    logger = logging.getLogger(__name__)

    load_dotenv()
    api_key = os.getenv("API_KEY", "not-set")
    masked = api_key[:3] + "*" * max(len(api_key) - 3, 0)
    logger.info(f"Loaded API_KEY from .env: {masked}")

    config = load_config()
    random.seed(config["training"]["random_state"])

    logger.info("Training started")
    df = load_data(config["data_path"])
    X = df.drop("label", axis=1)
    y = df["label"]

    model = RandomForestModel(
        n_estimators=config["model"]["n_estimators"],
        random_state=config["training"]["random_state"],
    )
    model.train(X, y)
    score = model.model.score(X, y)
    logger.info(f"Training complete. Accuracy: {score:.4f}")
    print(f"Training accuracy: {score:.4f}")


if __name__ == "__main__":
    main()
'''

with open("project/src/train.py", "w") as f:
    f.write(train_script)

print("Wrote project/src/train.py")


Wrote project/src/train.py


In [38]:
# Format and lint it, exactly as you would before committing
result = subprocess.run(["black", "project/src/train.py"], capture_output=True, text=True)
print("black:", result.stdout.strip() or result.stderr.strip())

result = subprocess.run(["ruff", "check", "project/src/train.py"], capture_output=True, text=True)
print("ruff check output:")
print(result.stdout if result.stdout.strip() else "No issues found.")


black: error: cannot format project\src\train.py: 'utf-8' codec can't decode byte 0x97 in position 14: invalid start byte

Oh no! \U0001f4a5 \U0001f494 \U0001f4a5
1 file failed to reformat.
ruff check output:
]8;;https://docs.astral.sh/ruff/rules/io-error\E902]8;;\ stream did not contain valid UTF-8
--> project\src\train.py:1:1

Found 1 error.



In [39]:
with open("project/src/train.py", "w", encoding="utf-8") as f:
    f.write(train_script)

print("Wrote project/src/train.py")

Wrote project/src/train.py


In [40]:
# Format and lint it, exactly as you would before committing
result = subprocess.run(["black", "project/src/train.py"], capture_output=True, text=True)
print("black:", result.stdout.strip() or result.stderr.strip())

result = subprocess.run(["ruff", "check", "project/src/train.py"], capture_output=True, text=True)
print("ruff check output:")
print(result.stdout if result.stdout.strip() else "No issues found.")


black: reformatted project\src\train.py

All done! \u2728 \U0001f370 \u2728
1 file reformatted.
ruff check output:
All checks passed!



In [41]:
# Run it exactly like a teammate would, from the terminal
result = subprocess.run(
    ["python", "src/train.py"],
    cwd="project",
    capture_output=True,
    text=True,
)
print("STDOUT:", result.stdout)
print("Exit code:", result.returncode)


STDOUT: Training accuracy: 1.0000

Exit code: 0


In [42]:
# Confirm the final project structure
def print_tree(path, prefix=""):
    entries = sorted(os.listdir(path))
    for i, entry in enumerate(entries):
        full = os.path.join(path, entry)
        connector = "\u2514\u2500\u2500 " if i == len(entries) - 1 else "\u251c\u2500\u2500 "
        print(prefix + connector + entry + ("/" if os.path.isdir(full) else ""))
        if os.path.isdir(full):
            extension = "    " if i == len(entries) - 1 else "\u2502   "
            print_tree(full, prefix + extension)

print("project/")
print_tree("project")


project/
├── .env
├── .env.example
├── .gitignore
├── configs/
│   └── config.yaml
├── data/
│   └── train.csv
├── logs/
│   └── train.log
├── models/
├── notebooks/
└── src/
    └── train.py


## Capstone: Refactor a Messy Legacy Script

Below is a messy legacy script — the kind you might inherit from a
teammate. It works, but it breaks nearly every practice from this
notebook: no functions or classes, a hardcoded path, no config, no error
handling, no logging, and no fixed random seed.

```python
# legacy_pipeline.py  (BEFORE)
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

data = pd.read_csv("/Users/someone/Desktop/final_data_v3_USE_THIS_ONE.csv")
data = data.dropna()
X = data.drop("label", axis=1)
y = data["label"]

clf = RandomForestClassifier(n_estimators=500)
clf.fit(X, y)
print("done, score:", clf.score(X, y))
```

**Your task:** apply everything from this notebook to fix it:
1. Move the hardcoded path and `n_estimators` into `configs/config.yaml`
2. Wrap the data loading logic in a class or typed functions
3. Add exception handling for a missing or empty dataset
4. Add logging instead of `print()`
5. Fix the random seed for reproducibility
6. Format and lint the result with Black and Ruff

Try it yourself in the cell below before checking the solution.


In [41]:
# TODO — your capstone solution here


In [43]:
# SOLUTION — this is exactly the project/src/train.py we already built above.
# Confirm it still runs end-to-end, formatted and linted, from the standard
# project structure:

result = subprocess.run(["python", "src/train.py"], cwd="project", capture_output=True, text=True)
print("STDOUT:", result.stdout)
print("Exit code:", result.returncode)

result = subprocess.run(["ruff", "check", "project/src/train.py"], capture_output=True, text=True)
print("ruff check:", result.stdout.strip() or "No issues found.")


STDOUT: Training accuracy: 1.0000

Exit code: 0
ruff check: All checks passed!


## Key Takeaways

- **Fundamentals are the foundation** — clean variables, control flow,
  and data structures underlie everything else.
- **Functions and classes organize logic** — modularity and OOP turn
  scripts into maintainable systems.
- **Robustness is not optional** — exception handling, logging, and
  configuration make code trustworthy outside of a notebook.
- **Structure and reproducibility tie it together** — the standard
  project layout, type hints, Black/Ruff, and fixed seeds are what make
  results shareable and repeatable.

Take this same `project/` skeleton and drop in your own dataset and
model — the structure, config, logging, and reproducibility pattern
stays exactly the same no matter what you're building.
